<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Agentic_RAG/Semantic_Chunking/Comparison_of_Different_Semantic_Chunking_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing Semantic Chunking Techniques

Before text can go into a RAG system, it has to be **chunked** — split into smaller pieces that get embedded and retrieved. *How* you split matters: cut in the wrong place and a chunk loses the context that makes it useful.

This notebook implements and compares **four semantic chunking strategies** on real SEC 10-K filings:

| Method | Core idea |
|---|---|
| **Statistical** | Cluster sentences by topic (KMeans), then pack into size-bounded chunks |
| **Consecutive** | Grow a chunk while each next sentence stays similar to it |
| **Cumulative** | Like consecutive, but compare against the *merged* chunk with length-weighted embeddings |
| **Double Merging** | Two passes — merge sentences, then merge adjacent similar chunks |

Each method is scored on **coherence**, **separation**, **size consistency**, and **speed** so we can see the trade-offs.

## Setup: install dependencies

First we install the libraries this notebook needs. `unstructured` parses documents, `sentence-transformers` produces embeddings, `transformers` supplies a tokenizer, and `scikit-learn` + `numpy` handle the clustering and vector math. (On Colab, you may need to restart the runtime after installing.)

In [ ]:
# Core dependencies:
#   unstructured            -> parse PDFs/docs into structured elements (text, tables, images)
#   sentence-transformers   -> turn sentences into embedding vectors
#   transformers            -> provides the GPT-2 tokenizer used below
#   scikit-learn            -> KMeans clustering for the statistical chunker
#   numpy                   -> vector math (cosine similarity, norms)
!pip install unstructured sentence-transformers transformers scikit-learn numpy

Reading PDFs needs extra parsing backends, so we install the `[pdf]` extra for `unstructured` separately. This pulls in tools (like `pdfminer` and `poppler`) that let `partition()` open and read `.pdf` files.

In [ ]:
# The [pdf] extra pulls in the PDF-specific parsing backends (e.g. pdfminer/poppler)
# that `unstructured.partition` needs to read .pdf files.
!pip install unstructured[pdf]

## Download the sample documents

We pull three real SEC **10-K annual reports** from the course repo into Colab's `/content` folder. These are long, dense, text-heavy filings — exactly the kind of document where chunking decisions show up clearly in the results.

In [ ]:
# Download the sample 10-K filings from the course repo into /content
# These are real SEC annual reports — long, text-heavy docs that make chunking differences visible.
BASE_URL = "https://raw.githubusercontent.com/hamzafarooq/multi-agent-course/main/modules/Module_3_Agentic_RAG/Semantic_Chunking/data"

# -q = quiet, -O = save under this exact filename (note the URL-encoded parentheses %28 %29).
!wget -q -O "/content/_10-K-2022-(As-Filed).pdf" "{BASE_URL}/_10-K-2022-%28As-Filed%29.pdf"
!wget -q -O "/content/0001047469-04-005840.pdf" "{BASE_URL}/0001047469-04-005840.pdf"
!wget -q -O "/content/10-K-netflix.pdf" "{BASE_URL}/10-K-netflix.pdf"

# Confirm the files landed and check their sizes.
!ls -lh /content/*.pdf

## Imports

Now we bring in everything the notebook uses: `numpy` for vector math, `time` for benchmarking, the tokenizer and embedding model, `KMeans` for clustering, and `unstructured`'s `partition` plus the element types (`Text`, `Table`, `Image`) we'll filter on.

In [ ]:
import numpy as np
import time
from typing import List, Dict
from transformers import AutoTokenizer            # tokenizer (token-level length awareness)
from sentence_transformers import SentenceTransformer  # sentence -> embedding vector
from sklearn.cluster import KMeans                # clustering for statistical chunking
from unstructured.partition.auto import partition # auto-detect file type and split into elements
from unstructured.documents.elements import Text, Table, Image  # element types we filter on

## Load the models

We load the models **once** and reuse them everywhere, since loading is the slow part. We use the lightweight **`all-MiniLM-L6-v2`** sentence-embedding model (fast, 384-dimensional vectors — a great default for chunking experiments) and the **GPT-2 tokenizer** (useful if you later want to measure chunk size in tokens rather than characters).

In [ ]:
# Initialize models once and reuse them (loading is expensive).
# GPT-2 tokenizer: handy if you want to measure chunk size in *tokens* rather than characters.
tokenizer = AutoTokenizer.from_pretrained("gpt2")
# all-MiniLM-L6-v2: small, fast sentence-embedding model (384-dim) — good default for chunking experiments.
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

## Helper 1: detect multimodal documents

`partition()` turns a file into a list of typed elements. This helper checks whether any of them are **tables or images** — i.e. whether the document is "multimodal." Our text-only chunking pipeline can't handle those well, so later we use this to skip such documents.

In [ ]:
def check_document_type(file_path: str) -> bool:
    """Check if the document is multimodal (contains tables or images)."""
    try:
        # partition() splits the file into a list of typed elements (Text, Table, Image, ...).
        elements = partition(filename=file_path)
        # Flag the doc as multimodal if ANY element is a table or image —
        # those need different handling than plain text, so we skip them later.
        return any(isinstance(element, (Table, Image)) for element in elements)
    except Exception as e:
        print(f"Error checking document type for {file_path}: {str(e)}")
        return False

## Helper 2: extract the text

This helper parses the document and keeps only the **`Text`** elements, concatenating them into one big string. Tables and images are dropped, leaving clean prose that the chunkers can split.

In [ ]:
def extract_text_content(file_path: str) -> str:
    """Extract only text content from the document."""
    try:
        elements = partition(filename=file_path)
        text_content = ""
        for element in elements:
            # Keep only the textual elements; drop tables/images so we chunk pure prose.
            if isinstance(element, Text):
                text_content += str(element) + "\n"
        return text_content
    except Exception as e:
        print(f"Error extracting text from {file_path}: {str(e)}")
        return ""

## Method 1 — Statistical (clustering-based) chunking

This is the first of our four strategies. It embeds every sentence, then uses **KMeans clustering** to group sentences into topics (roughly one cluster per 10 sentences). It then walks through the sentences and packs them into chunks, starting a new chunk whenever the size budget (`max_chunk_size`) would be exceeded.

Think of it as: *"figure out the topics statistically, then fill chunks up to a size limit."*

In [ ]:
def statistical_semantic_chunking(text: str, max_chunk_size: int = 512, overlap: int = 50) -> List[str]:
    """Statistical semantic chunking using sentence embeddings and clustering.

    Idea: embed every sentence, group them into topic clusters with KMeans,
    then emit chunks while respecting a max size budget.
    """
    # Naive sentence split on '.' — cheap, though it won't handle abbreviations perfectly.
    sentences = text.split('.')
    sentence_embeddings = embedding_model.encode(sentences)

    # Pick the number of topic clusters heuristically: ~1 cluster per 10 sentences (min 2).
    num_clusters = max(2, len(sentences) // 10)
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(sentence_embeddings)

    # Build chunks sentence-by-sentence, starting a new chunk once we'd exceed max_chunk_size.
    chunks = []
    current_chunk = ""
    for sentence, label in zip(sentences, labels):
        if len(current_chunk) + len(sentence) > max_chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += sentence + '.'

    # Don't forget the trailing chunk still being accumulated.
    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

## Method 2 — Consecutive chunking

Instead of clustering globally, this method walks sentences **in order**. It keeps a running chunk and compares each new sentence to the chunk's current embedding using **cosine similarity**. If the sentence is similar enough (and the chunk is still under the size limit), it gets appended and the embedding is averaged in; otherwise a topic shift is assumed and a new chunk begins.

Think of it as: *"keep adding sentences until the topic drifts."*

In [ ]:
def consecutive_semantic_chunking(text: str, max_chunk_size: int = 512, similarity_threshold: float = 0.8) -> List[str]:
    """Consecutive semantic chunking using sentence embeddings.

    Idea: walk sentences in order and keep extending the current chunk as long as the
    next sentence is similar enough to it; otherwise start a fresh chunk.
    """
    sentences = text.split('.')
    sentence_embeddings = embedding_model.encode(sentences)

    chunks = []
    # Seed the running chunk and its embedding with the first sentence.
    current_chunk = sentences[0]
    current_embedding = sentence_embeddings[0]

    for sentence, embedding in zip(sentences[1:], sentence_embeddings[1:]):
        # Cosine similarity between the running chunk's embedding and the next sentence.
        similarity = np.dot(current_embedding, embedding) / (np.linalg.norm(current_embedding) * np.linalg.norm(embedding))

        if similarity >= similarity_threshold and len(current_chunk) + len(sentence) <= max_chunk_size:
            # Similar and still under budget -> append, and average the embeddings to track the chunk's "center".
            current_chunk += '. ' + sentence
            current_embedding = (current_embedding + embedding) / 2
        else:
            # Topic shift (or size limit) -> close the chunk and start a new one.
            chunks.append(current_chunk.strip())
            current_chunk = sentence
            current_embedding = embedding

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

## Method 3 — Cumulative chunking

A refinement of consecutive chunking. Instead of comparing the next sentence directly, it compares the current chunk against the **prospective merged chunk**, using a **word-count-weighted average** of the embeddings. The weighting prevents a single short new sentence from swinging the chunk's "center" too much, so long chunks stay stable.

Think of it as: *"would merging this sentence change the chunk's meaning much? If not, merge it."*

In [ ]:
def cumulative_semantic_chunking(text: str, max_chunk_size: int = 512, similarity_threshold: float = 0.8) -> List[str]:
    """Cumulative semantic chunking using sentence embeddings.

    Like consecutive chunking, but compares the chunk against the *prospective merged*
    chunk and uses a word-count-weighted average so long chunks aren't dominated by one new sentence.
    """
    sentences = text.split('.')
    sentence_embeddings = embedding_model.encode(sentences)

    chunks = []
    current_chunk = sentences[0]
    current_embedding = sentence_embeddings[0]

    for sentence, embedding in zip(sentences[1:], sentence_embeddings[1:]):
        # Tentatively merge, and compute the merged embedding as a length-weighted average
        # (each sentence contributes in proportion to its word count).
        combined_chunk = current_chunk + '. ' + sentence
        combined_embedding = (current_embedding * len(current_chunk.split()) + embedding * len(sentence.split())) / (len(current_chunk.split()) + len(sentence.split()))

        # How much does adding this sentence shift the chunk's meaning? High similarity = little drift.
        similarity = np.dot(current_embedding, combined_embedding) / (np.linalg.norm(current_embedding) * np.linalg.norm(combined_embedding))

        if similarity >= similarity_threshold and len(combined_chunk) <= max_chunk_size:
            # Drift is small and we're under budget -> accept the merge.
            current_chunk = combined_chunk
            current_embedding = combined_embedding
        else:
            # Adding this sentence would change the topic too much -> start a new chunk.
            chunks.append(current_chunk.strip())
            current_chunk = sentence
            current_embedding = embedding

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

## Method 4 — Semantic double merging

The most aggressive strategy, run in **two passes**:

1. **Merge sentences** into initial chunks (same logic as consecutive chunking), keeping each chunk's embedding.
2. **Merge adjacent chunks** that are still similar to each other.

The second pass catches cases where related ideas got split into neighboring chunks during pass 1, producing fewer, more complete chunks.

Think of it as: *"merge sentences, then merge the chunks too."*

In [ ]:
def semantic_double_merging_chunking(text: str, max_chunk_size: int = 512, similarity_threshold: float = 0.8) -> List[str]:
    """Semantic double merging chunking using sentence embeddings.

    Two passes:
      1) merge consecutive sentences into initial chunks (same logic as consecutive chunking),
      2) merge adjacent *chunks* that are still similar — catches related ideas split across pass 1.
    """
    sentences = text.split('.')
    sentence_embeddings = embedding_model.encode(sentences)

    # ---- First pass: merge consecutive sentences ----
    initial_chunks = []
    current_chunk = sentences[0]
    current_embedding = sentence_embeddings[0]

    for sentence, embedding in zip(sentences[1:], sentence_embeddings[1:]):
        similarity = np.dot(current_embedding, embedding) / (np.linalg.norm(current_embedding) * np.linalg.norm(embedding))

        if similarity >= similarity_threshold and len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += '. ' + sentence
            current_embedding = (current_embedding + embedding) / 2
        else:
            # Keep the embedding alongside the text so the second pass can compare chunks.
            initial_chunks.append((current_chunk.strip(), current_embedding))
            current_chunk = sentence
            current_embedding = embedding

    if current_chunk:
        initial_chunks.append((current_chunk.strip(), current_embedding))

    # ---- Second pass: merge similar neighboring chunks ----
    final_chunks = []
    current_chunk, current_embedding = initial_chunks[0]

    for chunk, embedding in initial_chunks[1:]:
        similarity = np.dot(current_embedding, embedding) / (np.linalg.norm(current_embedding) * np.linalg.norm(embedding))

        if similarity >= similarity_threshold and len(current_chunk) + len(chunk) <= max_chunk_size:
            # Two adjacent chunks are still on-topic -> fuse them.
            current_chunk += ' ' + chunk
            current_embedding = (current_embedding + embedding) / 2
        else:
            final_chunks.append(current_chunk)
            current_chunk = chunk
            current_embedding = embedding

    if current_chunk:
        final_chunks.append(current_chunk)

    return final_chunks

## Scoring the chunks

To compare methods we need numbers. This function computes three quality metrics per set of chunks:

- **Coherence** — how internally consistent each chunk is. *(Caveat: as written it compares each chunk embedding with itself, so it's ~1.0 by construction; a stricter version would average similarity between the sentences inside a chunk.)*
- **Separation** — average cosine *distance* between chunks. Higher means chunks cover more distinct topics (less redundancy).
- **Size consistency** — standard deviation of chunk lengths. Lower means more uniform chunk sizes.

In [ ]:
def evaluate_chunking(chunks: List[str]) -> Dict[str, float]:
    """Evaluate the quality of the chunking."""
    # One embedding per chunk; used for both quality metrics below.
    embeddings = embedding_model.encode(chunks)

    # Coherence: average cosine similarity of each chunk's embedding with itself.
    # NOTE: a vector with itself is always 1.0, so this is ~1.0 by construction —
    # a real coherence metric would average pairwise similarity *between sentences inside* a chunk.
    coherence = np.mean([np.dot(emb, emb) / (np.linalg.norm(emb) ** 2) for emb in embeddings])

    # Separation: how distinct chunks are from one another (higher = more distinct topics).
    # Average cosine *distance* (1 - similarity) over every pair of chunks.
    if len(chunks) > 1:
        distances = []
        for i in range(len(embeddings)):
            for j in range(i+1, len(embeddings)):
                distance = 1 - np.dot(embeddings[i], embeddings[j]) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j]))
                distances.append(distance)
        separation = np.mean(distances)
    else:
        separation = 0

    # Size consistency: std-dev of chunk lengths (lower = more uniform chunk sizes).
    size_consistency = np.std([len(chunk) for chunk in chunks])

    return {
        "coherence": coherence,
        "separation": separation,
        "size_consistency": size_consistency
    }

## Running all four methods on one text

This driver runs the **same text** through every chunker, **times** each one, scores the resulting chunks, and records the chunk count. The output is a dictionary mapping each method name to its metrics — an apples-to-apples comparison.

In [ ]:
def compare_chunking_methods(text: str) -> Dict[str, Dict[str, float]]:
    # Run the same text through every chunker so we can compare them apples-to-apples.
    chunking_methods = [
        ("Statistical", statistical_semantic_chunking),
        ("Consecutive", consecutive_semantic_chunking),
        ("Cumulative", cumulative_semantic_chunking),
        ("Double Merging", semantic_double_merging_chunking)
    ]
    results = {}
    for method_name, method_func in chunking_methods:
        # print(f"Applying chunking method: {method_name}")
        # Time each method so we can weigh quality against speed.
        start_time = time.time()
        chunks = method_func(text)
        end_time = time.time()

        # print(f"Number of chunks: {len(chunks)}")
        # print(f"Average chunk length: {np.mean([len(chunk) for chunk in chunks])}")

        # Score the chunks, then attach runtime and chunk count to the metrics dict.
        evaluation = evaluate_chunking(chunks)
        evaluation["time"] = end_time - start_time
        evaluation["num_chunks"] = len(chunks)

        results[method_name] = evaluation

    return results

## Processing a single document end-to-end

This ties the pieces together for one file: skip it if it's multimodal, extract its text, guard against empty extraction, run the four-way comparison, and pretty-print the metrics. It returns the results dictionary so callers can aggregate across documents.

In [ ]:
def process_document(file_path: str):
    try:
        # Skip multimodal docs — this pipeline only handles plain text.
        is_multimodal = check_document_type(file_path)

        if is_multimodal:
            print(f"Document {file_path} is multimodal. Skipping processing.")
            return

        # Pull the text out, then guard against empty/failed extraction.
        text_content = extract_text_content(file_path)
        if not text_content:
            print(f"No text content extracted from {file_path}")
            return

        # Run all four chunkers and pretty-print their metrics.
        comparison_results = compare_chunking_methods(text_content)

        print(f"Chunking method comparison for {file_path}:")
        for method, results in comparison_results.items():
            print(f"  {method}:")
            for metric, value in results.items():
                print(f"    {metric}: {value:.4f}")

        return comparison_results

    except Exception as e:
        print(f"Error processing document {file_path}: {str(e)}")
        return None

## Run it: batch over all documents

Finally we run the comparison across all three filings. `process_multiple_documents()` loops the paths through `process_document()`, aggregates each method's metrics across documents, and reports an **optimal method** based on the trade-off between separation, size consistency, and speed.

> **Note:** the `process_multiple_documents()` function is referenced here but isn't defined in a cell above — define it (a loop over `process_document` that averages the per-method metrics) before running this cell.

In [ ]:
if __name__ == "__main__":
    # Batch entry point: run the comparison over several filings at once.
    # NOTE: process_multiple_documents() should loop these paths through process_document()
    # and aggregate the per-method metrics across documents.
    file_paths = [
        "/content/_10-K-2022-(As-Filed).pdf",
        "/content/0001047469-04-005840.pdf",
        "/content/10-K-netflix.pdf"

    ]
    process_multiple_documents(file_paths)

## A second run with a different document set

The same batch run, swapping the third filing for another 10-K. Comparing the two aggregate outputs shows how stable the "optimal method" choice is across different document mixes — a quick sanity check that the winner isn't just an artifact of one particular file.

In [ ]:
if __name__ == "__main__":
    # Same batch run, swapping in a different third filing.
    # The aggregate output picks an "Optimal method" by trading off separation, size consistency, and speed.
    file_paths = [
        "/content/_10-K-2022-(As-Filed).pdf",
        "/content/0001047469-04-005840.pdf",
        "/content/0001731122-22-000634.pdf"

    ]
    process_multiple_documents(file_paths)